# YOLOv10n 1차 학습 (HaGRID 데이터셋)

전제: `prepare_hagrid_dataset.py` 실행 완료 후 `./data/hagrid_yolo/dataset.yaml`이 생성된 상태.

이 노트북은 COCO 사전학습 YOLOv10n 가중치를 가져와서 HaGRID 9개 클래스로 1차 학습함.
이후 직접 수집한 노인 손 데이터로 2차 fine-tuning은 별도 노트북에서 진행.

## 0. 환경 설정

In [ ]:
!pip install ultralytics

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
# GPU 사용 가능 안 뜨면: 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 다시 실행

## 1. dataset.yaml 확인

In [ ]:
DATASET_YAML = "./data/hagrid_yolo/dataset.yaml"

with open(DATASET_YAML, "r") as f:
    print(f.read())

## 2. 학습 설정

필요하면 아래 값들만 수정하면 됨.

In [ ]:
# ============================================================
# CONFIG — 여기만 수정하면 됨
# ============================================================

EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16          # GPU 메모리 부족하면 8로 낮출 것
PRETRAINED_WEIGHTS = "yolov10n.pt"   # COCO 사전학습 가중치, 자동 다운로드됨
RUN_NAME = "hagrid_stage1"
PROJECT_DIR = "./runs"

# ============================================================

## 3. 모델 로드 및 학습

In [ ]:
from ultralytics import YOLO

model = YOLO(PRETRAINED_WEIGHTS)

In [ ]:
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=RUN_NAME,
    project=PROJECT_DIR,
    patience=15,        # 15 epoch 동안 개선 없으면 조기 종료
    save=True,
    plots=True,
)

## 4. 학습 결과 확인

In [ ]:
import os

weights_dir = f"{PROJECT_DIR}/{RUN_NAME}/weights"
print("가중치 파일 목록:")
for f in os.listdir(weights_dir):
    print(" -", f)

print(f"\n최종 가중치 경로 (다음 단계 fine-tuning에 사용):")
print(f"{weights_dir}/best.pt")

In [ ]:
from IPython.display import Image as IPImage

# 학습 결과 그래프 (loss, mAP 등)
IPImage(filename=f"{PROJECT_DIR}/{RUN_NAME}/results.png")

In [ ]:
# 혼동 행렬 (어떤 클래스끼리 헷갈리는지 확인용 — three/three2, three_gun 등 체크)
IPImage(filename=f"{PROJECT_DIR}/{RUN_NAME}/confusion_matrix.png")

## 5. 검증셋 성능 평가

In [ ]:
best_model = YOLO(f"{weights_dir}/best.pt")
metrics = best_model.val(data=DATASET_YAML)

print("\n=== 검증 결과 ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 6. 샘플 이미지로 추론 테스트

In [ ]:
import glob

# 테스트셋에서 이미지 몇 장 골라서 추론 확인
test_images = glob.glob("./data/hagrid_yolo/images/test/*.jpg")[:5]

if test_images:
    test_results = best_model.predict(test_images, save=True, conf=0.4)
    print(f"\n추론 결과 저장 위치: {test_results[0].save_dir}")
else:
    print("테스트 이미지를 찾지 못함. 경로를 확인할 것.")

In [ ]:
# 추론된 이미지 한 장 미리보기
if test_images:
    import os
    saved_files = os.listdir(test_results[0].save_dir)
    if saved_files:
        IPImage(filename=os.path.join(test_results[0].save_dir, saved_files[0]))

## 7. 가중치 다운로드 (Google Drive 백업용)

코랩 런타임 종료되면 `./runs` 폴더가 날아가므로, 다음 fine-tuning 단계를 위해
`best.pt`는 반드시 Drive나 로컬로 백업할 것.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/HandSpark/weights"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(f"{weights_dir}/best.pt", f"{DRIVE_SAVE_DIR}/hagrid_stage1_best.pt")
print(f"저장됨: {DRIVE_SAVE_DIR}/hagrid_stage1_best.pt")